# 第四课：AI系统评估实践 —— 从「能跑」到「可靠」

## 学习目标
- 理解 AI 系统评估的四个核心维度
- 用代码批量测试不同模型，系统化地对比它们的表现
- 学会模型选择的工作流程：从需求到决策
- 为真实场景设计一份可复用的评估指南（Evaluation Guide）
>
> 公共基准测试（MMLU / Chatbot Arena 等）的意义与局限属于概念轨，见教程文档课程四。

> 单个回答的好坏容易判断，但整个系统的可靠性需要系统化的评估方法。

## 环境准备

> 请先运行 `00_Environment_Setup.ipynb` 完成环境配置（安装依赖包 + 设置 API Key），
> 然后再回到本 Notebook。

完成后，运行下面的代码加载环境变量：

In [ ]:
# 从 .env 文件加载 API Key（无需每次输入）
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

# ===== 选一个服务商：只改这一行，其他都不用动 =====
#   'openai'     云端  需要 OPENAI_API_KEY      效果最强，支持 Embedding
#   'deepseek'   云端  需要 DEEPSEEK_API_KEY    云端最便宜，无 Embedding
#   'openrouter' 云端  需要 OPENROUTER_API_KEY  可调用多家模型，无 Embedding
#   'ollama'     本地  不需要 Key，免费离线     先跑 `ollama serve` 并 pull 模型
PROVIDER = 'openai'

# 下面四家都兼容 OpenAI 的接口格式，区别只在：地址、Key、模型名。
PROVIDERS = {
    'openai': {
        'base_url': None,                            # None = 用 OpenAI 官方默认地址
        'api_key': os.getenv('OPENAI_API_KEY'),
        'model': 'gpt-5.6-luna',                     # 小模型：便宜、快
        'model_big': 'gpt-5.6-terra',                # 大模型：贵、强
        'embedding_model': 'text-embedding-3-small',
    },
    'deepseek': {
        'base_url': 'https://api.deepseek.com/v1',
        'api_key': os.getenv('DEEPSEEK_API_KEY'),
        'model': 'deepseek-v4-flash',                # 快、便宜
        'model_big': 'deepseek-v4-pro',              # 更强、更慢；V4 两个模型都会先思考再回答
        'embedding_model': None,                     # DeepSeek 目前不提供 Embedding 接口
    },
    'openrouter': {
        'base_url': 'https://openrouter.ai/api/v1',
        'api_key': os.getenv('OPENROUTER_API_KEY'),
        'model': 'openai/gpt-5.6-luna',
        'model_big': 'openai/gpt-5.6-terra',
        'embedding_model': None,                     # OpenRouter 不转发 Embedding 接口
    },
    'ollama': {
        'base_url': os.getenv('OLLAMA_BASE_URL', 'http://localhost:11434/v1'),
        'api_key': 'ollama',                         # 本地模型不校验 Key
        'model': 'gemma4:e2b-mlx',                   # 需先 ollama pull gemma4:e2b-mlx
        'model_big': 'gemma4:e2b-mlx',
        'embedding_model': 'nomic-embed-text',       # 需先 ollama pull nomic-embed-text
    },
}

cfg = PROVIDERS[PROVIDER]

# 先检查 Key：Key 为空时 OpenAI 客户端会直接抛出一长串报错，不容易看懂。
if not cfg['api_key']:
    raise SystemExit(
        f"没读到 '{PROVIDER}' 的 API Key。请在 .env 文件里补上 {PROVIDER.upper()}_API_KEY，\n"
        f"或者把上面的 PROVIDER 改成 'ollama'，用本地模型运行，完全不需要 Key。"
    )

client = OpenAI(api_key=cfg['api_key'], base_url=cfg['base_url'])

# 后面所有代码都只用这三个变量，换服务商不需要改任何一行业务代码
MODEL = cfg['model']
MODEL_BIG = cfg['model_big']
EMBEDDING_MODEL = cfg['embedding_model']

print(f'连接成功！服务商 = {PROVIDER}，默认模型 = {MODEL}')


---

## 活动一：用你的需求「面试」不同的 AI 模型

### 活动目标
设计与你工作/学习相关的真实任务，用相同的输入测试不同模型，系统化地打分对比。
这就像「面试」多个候选人，看谁最适合你的需求。

In [ ]:
# 活动一：批量模型对比

# 定义你的3个测试任务（请根据自己的实际情况修改）
test_tasks = [
    {
        'name': '信息总结',
        'prompt': '请将以下会议记录总结为3个要点，每个要点不超过50字：\n'
                 '今天团队讨论了Q3产品路线图。首先，产品经理提出了3个新功能需求，'
                 '包括用户画像优化、智能推荐升级和移动端适配。技术负责人表示'
                 '智能推荐需要额外2周开发时间。市场部建议优先做移动端适配，'
                 '因为移动端用户占比已达70%。最终决定：Q3优先移动端适配和用户画像优化，'
                 '智能推荐延至Q4。'
    },
    {
        'name': '创意写作',
        'prompt': '请用200字写一篇产品发布会的邀请函，产品是「AI智能笔记本」，'
                 '目标受众是商务人士，语气要专业但不失热情。'
    },
    {
        'name': '逻辑推理',
        'prompt': '小明说：「如果明天下雨，我就不去公园。」'
                 '第二天小明去了公园。请问：第二天有没有下雨？请分析推理过程。'
    }
]

# 要测试的模型列表
models = list(dict.fromkeys([MODEL, MODEL_BIG]))  # 没有权限的模型会自动跳过，不会中断

# 评估维度
dimensions = ['准确性(1-5)', '完整性(1-5)', '清晰度(1-5)', '有用性(1-5)']

print('开始批量评估...\n')
all_results = {}

for model_name in models:
    print(f'====== 测试模型: {model_name} ======')
    model_out = {}
    for task in test_tasks:
        try:
            r = client.chat.completions.create(
                model=model_name,
                messages=[{'role':'user','content':task['prompt']}],
                temperature=0.5)
        except Exception as e:
            # 没有该模型权限时跳过，不影响其他模型的对比
            print(f'[跳过] {model_name} 不可用：{str(e)[:100]}')
            model_out = {}
            break
        model_out[task['name']] = r.choices[0].message.content
        print(f'\n--- {task["name"]} ---')
        print(r.choices[0].message.content[:200] + '...')
    if model_out:
        all_results[model_name] = model_out

print('\n所有回答已获取完毕！请在下方评分。')

### 评估打分表

请在下面的代码单元中，对每个模型的每个回答打分：

In [ ]:
# 请根据上面的输出，填写你的评分
# 格式：scores[模型名][任务名] = [准确性, 完整性, 清晰度, 有用性]

scores = {
    MODEL: {
        '信息总结': [4, 5, 5, 4],  # 请根据实际修改
        '创意写作': [4, 4, 4, 3],
        '逻辑推理': [5, 4, 4, 4],
    },
    MODEL_BIG: {
        '信息总结': [5, 5, 5, 5],  # 请根据实际修改
        '创意写作': [5, 4, 5, 4],
        '逻辑推理': [5, 5, 5, 5],
    },
}

# 只统计上一步真正拿到回答的模型（没权限的模型会被自动剔除）
scores = {m: s for m, s in scores.items() if m in all_results}
if not scores:
    print('上一步没有任何模型跑通，请先检查 API Key 和 PROVIDER 设置。')

# 计算汇总分数
print(f'{"模型":<20}{"信息总结":<10}{"创意写作":<10}{"逻辑推理":<10}{"总分":<10}')
print('-' * 60)
for model, tasks in scores.items():
    task_totals = []
    row = f'{model:<20}'
    total = 0
    for task_name, sc in tasks.items():
        s = sum(sc)
        task_totals.append(s)
        total += s
        row += f'{s:<10}'
    row += f'{total:<10}'
    print(row)

# 找出最佳模型
if scores:
    best_model = max(scores, key=lambda m: sum(sum(v) for v in scores[m].values()))
    print(f'\n最佳模型: {best_model}')

### 讨论
- 不同模型在不同任务上的表现有差异吗？
- 如果只能选一个模型日常使用，你选哪个？
- 你的「评分」和「感觉」一致吗？有时候我们觉得某个模型「更好」，但打分却显示另一个更高。

---

## 活动二：让 AI 帮你做批量评估

### 活动目标
上面的手工评估很费时间。现在让 AI 帮你做批量评估——给 AI 一套标准，让它自动打分。

In [ ]:
# 活动二：AI 自动批量评估

eval_template = (
    '请对以下AI回答进行评分（每项1-5分）：\n'
    '1. 准确性：信息是否准确无误\n'
    '2. 完整性：是否覆盖所有要点\n'
    '3. 清晰度：是否清晰易懂\n'
    '4. 有用性：对用户有多大帮助\n\n'
    '请严格评分，不要心软。输出格式：\n'
    '准确性: X/5, 完整性: X/5, 清晰度: X/5, 有用性: X/5, 总分: X/20'
)

# 对每个任务进行AI自动评估
print('AI 自动评估结果：\n')
for model_name, tasks in all_results.items():
    print(f'====== {model_name} ======')
    for task_name, answer in tasks.items():
        print(f'\n--- {task_name} ---')
        r = client.chat.completions.create(
            model=MODEL,
            messages=[
                {'role':'system','content':'你是严格的AI评估专家。'},
                {'role':'user','content':f'{eval_template}\n\n任务：{task_name}\nAI回答：{answer}'}
            ],
            temperature=0.2)
        print(r.choices[0].message.content)

print('\n对比你的手工评分和AI自动评分，差异大吗？')

### 讨论
- AI 自动评估的结果和你的手工评分一致吗？
- 如果你有100个回答需要评估，你会怎么做？
- AI 做裁判的优缺点分别是什么？

---

## 活动三：设计你的「评估指南」

### 活动目标
假设你要搭建一个「AI客服」，设计一份完整的评估指南。这个练习帮助你理解评估的系统性。

In [ ]:
# 活动三：为「AI客服」设计评估指南

# 你的5条评估标准
eval_guide = '''
AI客服评估指南 v1.0
===================

标准1：意图理解（1-5分）
  1分 = 完全误解客户意图
  3分 = 大致理解，但有关键偏差
  5分 = 精准理解，甚至捕捉到隐含需求

标准2：信息准确性（1-5分）
  1分 = 提供错误信息
  3分 = 基本正确，有模糊之处
  5分 = 信息精确、完整，有据可查

标准3：语气友善度（1-5分）
  1分 = 冷漠生硬，像机器人
  3分 = 礼貌但缺乏温度
  5分 = 温暖自然，让人感到被重视

标准4：解决效率（1-5分）
  1分 = 完全没有解决问题
  3分 = 部分解决，需要客户额外操作
  5分 = 一次性解决，超出客户预期

标准5：安全合规（1-5分）
  1分 = 包含敏感信息泄露或不当承诺
  3分 = 基本安全，但措辞可改进
  5分 = 完全合规，且主动引导正确方向
'''

print(eval_guide)

# 让 AI 评价这份评估指南
r = client.chat.completions.create(
    model=MODEL,
    messages=[{'role':'user','content':f'请评价这份AI客服评估指南，指出它的优点和可以改进的地方，并提出2-3条具体的改进建议：\n{eval_guide}'}],
    temperature=0.5)
print('AI 对评估指南的反馈：')
print(r.choices[0].message.content)

---

## 本节回顾

| 技能 | 说明 |
|------|------|
| 批量模型对比 | 用相同输入系统化测试多个模型 |
| AI 自动评估 | 用 AI 做批量评估，高效但需谨慎 |
| 评估指南设计 | 为实际场景设计完整的评估框架 |

### 课后练习
1. 将你的评估指南用于实际评估3个 AI 回答，看看标准是否好用
2. 访问 lmarena.ai 浏览 Chatbot Arena 排行榜
3. 思考：如果你要为一个 AI 应用设计评估体系，你会从哪几个维度入手？